# Antibiotic-Group Scientific Interest in AMR

This notebook generalizes the yearly AMR scientific-interest workflow from individual antibiotics to a coarser antibiotic abstraction.

The user chooses one grouping column from `archive/legacy_assets/antibiotic_list2.xlsx`, for example:

- `Antibiotic`
- `Class`
- `Simplified categories`
- `High level categories`
- or any other column present in the spreadsheet

The workflow then:

1. maps each antibiotic to the chosen group
2. aggregates yearly AMR-positive counts from `(pathogen, antibiotic, year)` to `(pathogen, group, year)`
3. aggregates antibiotic background counts to the same group level
4. runs the same yearly SI modeling on `(pathogen, antibiotic_group, year)` pairs

Internally, the coarser group is renamed to the `antibiotic` column expected by the shared scientific-interest functions.


In [15]:
%config InlineBackend.figure_format = 'retina'

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge

PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "amr").exists():
        PROJECT_ROOT = candidate
        break
    nested_candidate = candidate / "Antimicrobial Resistance"
    if (nested_candidate / "amr").exists():
        PROJECT_ROOT = nested_candidate
        break

if PROJECT_ROOT is None:
    raise ModuleNotFoundError("Could not locate the project root containing the 'amr' package.")

ARCHIVE_ROOT = PROJECT_ROOT / "archive"
if str(ARCHIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(ARCHIVE_ROOT))

from amr_legacy.scientific_interest import (
    plot_yearly_si_pair,
    display_si_amr_pareto_front,
    plot_yearly_si_prediction_for_pareto_pairs,
    rank_yearly_pairs_by_future_si_shift,
    select_interesting_yearly_rank_cases,
)


## Configuration

Set `ANTIBIOTIC_GROUP_COLUMN` to any column present in `archive/legacy_assets/antibiotic_list2.xlsx`.
The selected grouping column becomes the coarse antibiotic dimension for all downstream analysis.


In [16]:
ARCHIVE_DIR = PROJECT_ROOT / "archive"
LEGACY_ASSETS_DIR = ARCHIVE_DIR / "legacy_assets"

ANTIBIOTIC_METADATA_PATH = LEGACY_ASSETS_DIR / "antibiotic_list2.xlsx"
REFINED_SCREEN_PATH = LEGACY_ASSETS_DIR / "pmids_results_positive_yearly_refined_screen.parquet"
REFINED_PAIR_PMID_PATH = LEGACY_ASSETS_DIR / "pmids_results_positive_yearly_refined_pair_pmid.parquet"
YEARLY_PATH = LEGACY_ASSETS_DIR / "pmids_results_positive_yearly_refined.parquet"
PATHOGEN_BG_PATH = PROJECT_ROOT / "pathogen_tiab_counts.parquet"
ANTIBIOTIC_BG_PATH = PROJECT_ROOT / "antibiotic_tiab_counts.parquet"

ANTIBIOTIC_GROUP_COLUMN = "Antibiotic"
N_ROWS = 10
SHOW_RANDOM_ROWS = True

RANK_FIT_START_YEAR = 1970
RANK_FIT_END_YEAR = 2022
RANK_NUM_FUTURE_YEARS = 20

RANK_OUTPUT_PATH = LEGACY_ASSETS_DIR / (
    f"si_yearly_rank_shift_{ANTIBIOTIC_GROUP_COLUMN.lower().replace(' ', '_')}_fit_{RANK_FIT_START_YEAR}_{RANK_FIT_END_YEAR}_ref_{RANK_FIT_END_YEAR}_future_{RANK_FIT_END_YEAR + RANK_NUM_FUTURE_YEARS}.parquet"
)
PARETO_PNG_DIR = LEGACY_ASSETS_DIR / f"pareto_pngs_yearly_{ANTIBIOTIC_GROUP_COLUMN.lower().replace(' ', '_')}"
INTERESTING_PNG_DIR = LEGACY_ASSETS_DIR / f"interesting_yearly_si_pngs_{ANTIBIOTIC_GROUP_COLUMN.lower().replace(' ', '_')}"


In [17]:
antibiotics_df = pd.read_excel(ANTIBIOTIC_METADATA_PATH)
print("Available grouping columns:")
for col in antibiotics_df.columns:
    n_unique = pd.Series(antibiotics_df[col]).dropna().astype(str).nunique()
    print(f"- {col}: {n_unique}")

if ANTIBIOTIC_GROUP_COLUMN not in antibiotics_df.columns:
    raise KeyError(
        f"ANTIBIOTIC_GROUP_COLUMN={ANTIBIOTIC_GROUP_COLUMN!r} is not in antibiotic metadata columns {list(antibiotics_df.columns)}"
    )

sample_n = min(N_ROWS, len(antibiotics_df))
sample_df = antibiotics_df.sample(sample_n, random_state=None) if SHOW_RANDOM_ROWS else antibiotics_df.head(sample_n)
display(sample_df)


Available grouping columns:
- Antibiotic: 245
- Class: 39
- Simplified categories: 15
- High level categories: 11
- Origin: 2


,Antibiotic,Class,Simplified categories,High level categories,Origin
216,Sulfamethoxazole/trimethoprim,Sulfonamide-trimethoprim-combinations,Sulfonamides/Trimethoprim derivatives and their combinations,Sulfonamides/Trimethoprim derivatives and their combinations,Synthetic
70,Ceftolozane/tazobactam,Fifth-generation cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
73,Flomoxef,Second-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
97,Carindacillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified
241,Penimepicycline,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
168,Enoxacin,Fluoroquinolones,Qinolones/fluoroquinolones,Qinolones/fluoroquinolones,Synthetic
139,Miocamycin,Macrolides,Macrolides,MLS,Naturally occurring/modified
10,Plazomicin,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
193,Temafloxacin,Fluoroquinolones,Qinolones/fluoroquinolones,Qinolones/fluoroquinolones,Synthetic
109,Oxacillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified


## Build the antibiotic-to-group mapping

The mapping uses `Antibiotic` as the fine-grained source key and the user-selected column as the target group label.


In [18]:
mapping_df = antibiotics_df[["Antibiotic", ANTIBIOTIC_GROUP_COLUMN]].copy()
mapping_df = mapping_df.rename(columns={ANTIBIOTIC_GROUP_COLUMN: "antibiotic_group"})
mapping_df["Antibiotic"] = mapping_df["Antibiotic"].astype(str).str.strip()
mapping_df["antibiotic_group"] = mapping_df["antibiotic_group"].astype(str).str.strip()
mapping_df = mapping_df.replace({"": np.nan}).dropna(subset=["Antibiotic", "antibiotic_group"])

duplicate_mapping_df = (
    mapping_df.groupby("Antibiotic")["antibiotic_group"]
    .nunique()
    .reset_index(name="n_groups")
)
duplicate_mapping_df = duplicate_mapping_df.loc[duplicate_mapping_df["n_groups"] > 1]
if not duplicate_mapping_df.empty:
    raise ValueError(
        f"Some antibiotics map to multiple {ANTIBIOTIC_GROUP_COLUMN!r} values. Fix the metadata first."
    )

mapping_df = mapping_df.drop_duplicates(subset=["Antibiotic"]).reset_index(drop=True)
print(f"Mapped {len(mapping_df):,} antibiotics into {mapping_df['antibiotic_group'].nunique():,} groups using column {ANTIBIOTIC_GROUP_COLUMN!r}.")
display(mapping_df.sample(min(N_ROWS, len(mapping_df)), random_state=None))


KeyError: 'Antibiotic'

## Load yearly AMR counts and yearly backgrounds at the chosen group level

This cell remaps yearly AMR-positive rows and antibiotic background counts from individual antibiotics to the selected coarse grouping.


In [ ]:
yearly_df_raw = pd.read_parquet(YEARLY_PATH)
pathogen_bg_df = pd.read_parquet(PATHOGEN_BG_PATH).rename(columns={"count": "bg_pathogen_n"})
antibiotic_bg_df_raw = pd.read_parquet(ANTIBIOTIC_BG_PATH)

yearly_df_grouped = (
    yearly_df_raw.merge(mapping_df, left_on="antibiotic", right_on="Antibiotic", how="left")
)
amr_mapping_coverage = yearly_df_grouped["antibiotic_group"].notna().mean()
print(f"AMR yearly mapping coverage: {100 * amr_mapping_coverage:.2f}%")

yearly_df_grouped = yearly_df_grouped.dropna(subset=["antibiotic_group"]).copy()
yearly_df_grouped["pmids"] = yearly_df_grouped["pmids"].apply(lambda x: x if isinstance(x, list) else [])
yearly_df_grouped = (
    yearly_df_grouped.groupby(["pathogen", "antibiotic_group", "year"], as_index=False)
    .agg(
        count=("count", "sum"),
        pmids=("pmids", lambda lists: list(dict.fromkeys(pmid for sub in lists for pmid in sub))),
    )
)

antibiotic_bg_df = antibiotic_bg_df_raw.merge(mapping_df, left_on="antibiotic", right_on="Antibiotic", how="left")
bg_mapping_coverage = antibiotic_bg_df["antibiotic_group"].notna().mean()
print(f"Antibiotic background mapping coverage: {100 * bg_mapping_coverage:.2f}%")

antibiotic_bg_df = antibiotic_bg_df.dropna(subset=["antibiotic_group"]).copy()
antibiotic_bg_df = (
    antibiotic_bg_df.groupby(["antibiotic_group", "year"], as_index=False)["count"]
    .sum()
    .rename(columns={"count": "bg_antibiotic_n"})
)

amr_yearly_df = (
    yearly_df_grouped.merge(pathogen_bg_df, on=["pathogen", "year"], how="left")
    .merge(antibiotic_bg_df, on=["antibiotic_group", "year"], how="left")
    .rename(columns={"antibiotic_group": "antibiotic"})
    .copy()
)

amr_yearly_df["count"] = pd.to_numeric(amr_yearly_df["count"], errors="coerce")
amr_yearly_df["bg_pathogen_n"] = pd.to_numeric(amr_yearly_df["bg_pathogen_n"], errors="coerce")
amr_yearly_df["bg_antibiotic_n"] = pd.to_numeric(amr_yearly_df["bg_antibiotic_n"], errors="coerce")
amr_yearly_df["log_count"] = np.log10(1 + amr_yearly_df["count"])
amr_yearly_df["log_bg_pathogen_n"] = np.log10(1 + amr_yearly_df["bg_pathogen_n"])
amr_yearly_df["log_bg_antibiotic_n"] = np.log10(1 + amr_yearly_df["bg_antibiotic_n"])
amr_yearly_df = amr_yearly_df.dropna(
    subset=["count", "bg_pathogen_n", "bg_antibiotic_n", "log_count", "log_bg_pathogen_n", "log_bg_antibiotic_n"]
).reset_index(drop=True)

print(f"grouped yearly AMR rows: {len(amr_yearly_df):,}")
print(f"unique pathogen-{ANTIBIOTIC_GROUP_COLUMN.lower()} pairs: {amr_yearly_df[['pathogen', 'antibiotic']].drop_duplicates().shape[0]:,}")
print(f"unique {ANTIBIOTIC_GROUP_COLUMN.lower()} values in modeled data: {amr_yearly_df['antibiotic'].nunique():,}")
print(f"year span: {int(amr_yearly_df['year'].min())} to {int(amr_yearly_df['year'].max())}")


In [ ]:
sample_n = min(N_ROWS, len(amr_yearly_df))
sample_df = amr_yearly_df.sample(sample_n, random_state=None) if SHOW_RANDOM_ROWS else amr_yearly_df.head(sample_n)
display(sample_df)


## Pooled yearly baseline model at the group level

The model is unchanged except that the antibiotic dimension is now the selected coarse grouping.


In [ ]:
alpha = 1.0

pathogen_dummies = pd.get_dummies(amr_yearly_df["pathogen"].astype(str), prefix="pathogen", dtype=float)
antibiotic_dummies = pd.get_dummies(amr_yearly_df["antibiotic"].astype(str), prefix="antibiotic", dtype=float)

design_df = pd.concat(
    [
        amr_yearly_df[["log_bg_pathogen_n", "log_bg_antibiotic_n"]].reset_index(drop=True),
        pathogen_dummies.reset_index(drop=True),
        antibiotic_dummies.reset_index(drop=True),
    ],
    axis=1,
)

response = amr_yearly_df["log_count"].to_numpy(dtype=float)
pooled_model = Ridge(alpha=alpha, fit_intercept=True)
pooled_model.fit(design_df, response)

si_df = amr_yearly_df.copy()
si_df["expected_log_count_global"] = pooled_model.predict(design_df)
si_df["expected_count_global"] = np.clip(np.power(10.0, si_df["expected_log_count_global"]) - 1.0, 0.0, None)
si_df["SI"] = si_df["log_count"] - si_df["expected_log_count_global"]

display(
    si_df[
        [
            "pathogen",
            "antibiotic",
            "year",
            "count",
            "bg_pathogen_n",
            "bg_antibiotic_n",
            "expected_count_global",
            "SI",
        ]
    ].head(N_ROWS)
)


In [ ]:
coef_df = pd.DataFrame({"feature": design_df.columns, "coefficient": pooled_model.coef_})
display(coef_df[coef_df["feature"].isin(["log_bg_pathogen_n", "log_bg_antibiotic_n"])])
print("Model intercept:", float(pooled_model.intercept_))


## Rank pathogen-group pairs by average yearly scientific interest


In [ ]:
pair_summary_df = (
    si_df.groupby(["pathogen", "antibiotic"], as_index=False)
    .agg(
        mean_SI=("SI", "mean"),
        total_count=("count", "sum"),
        mean_count=("count", "mean"),
        n_years=("year", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .sort_values(["mean_SI", "total_count"], ascending=[False, False])
    .reset_index(drop=True)
)

display(pair_summary_df.head(20))


## Plot one yearly pathogen-group trajectory

Choose any pathogen and any coarse antibiotic group present in `si_df`.


In [ ]:
MIN_TRAIN_YEARS = 10
MIN_TOTAL_AMR_COUNT = 200

eligible_pairs = (
    si_df.loc[si_df["year"].between(RANK_FIT_START_YEAR, RANK_FIT_END_YEAR)]
    .groupby(["pathogen", "antibiotic"], as_index=False)
    .agg(
        n_train_years=("year", "nunique"),
        total_count=("count", "sum"),
    )
    .query("n_train_years >= @MIN_TRAIN_YEARS and total_count >= @MIN_TOTAL_AMR_COUNT")
    .reset_index(drop=True)
)

if eligible_pairs.empty:
    raise ValueError(
        f"No pathogen-group pairs have at least {MIN_TRAIN_YEARS} training years "
        f"and total AMR count >= {MIN_TOTAL_AMR_COUNT}."
    )

example_row = eligible_pairs.sample(1, random_state=None).iloc[0]
example_pathogen = example_row["pathogen"]
example_group = example_row["antibiotic"]

print(
    f"Random eligible pair: {example_pathogen} vs {example_group} "
    f"(n_train_years={example_row['n_train_years']}, total_count={example_row['total_count']})"
)


result = plot_yearly_si_pair(
    si_df=si_df,
    pathogen=example_pathogen,
    antibiotic=example_group,
    fit_start_year=RANK_FIT_START_YEAR,
    fit_end_year=RANK_FIT_END_YEAR,
    num_additional_years=RANK_NUM_FUTURE_YEARS,
    loess_frac=0.6,
    loess_degree=2,
    tail_damping_years=None,
)

print(f"Example plot for {example_pathogen} vs {example_group} ({ANTIBIOTIC_GROUP_COLUMN})")
print("Observed vs predicted yearly AMR counts")
display(result["comparison_df"])
print("LOESS fraction:", result["loess_fit"]["frac"])
print("LOESS degree:", result["loess_fit"]["degree"])
print("Training residual std on log-count scale:", result["loess_fit"]["residual_std"])
print("Latest observed year:", result["latest_available_year"])


## Pareto-front examples at the group level


In [ ]:
n_pareto_shells = 6
pareto_df = display_si_amr_pareto_front(si_df, n_pareto_shells=n_pareto_shells)
pareto_plot_results = plot_yearly_si_prediction_for_pareto_pairs(
    si_df,
    pareto_df,
    fit_start_year=RANK_FIT_START_YEAR,
    fit_end_year=RANK_FIT_END_YEAR,
    num_additional_years=RANK_NUM_FUTURE_YEARS,
    loess_frac=0.6,
    loess_degree=2,
    tail_damping_years=None,
    n_pareto_shells=n_pareto_shells,
    save_dir=str(PARETO_PNG_DIR),
)


## Rank future SI shifts for pathogen-group pairs


In [ ]:
%%time

rank_df = rank_yearly_pairs_by_future_si_shift(
    si_df,
    fit_start_year=RANK_FIT_START_YEAR,
    fit_end_year=RANK_FIT_END_YEAR,
    num_future_years=RANK_NUM_FUTURE_YEARS,
    save_path=str(RANK_OUTPUT_PATH),
    verbose=True,
)


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

rank_df = pd.read_parquet(RANK_OUTPUT_PATH)

interesting_df = select_interesting_yearly_rank_cases(
    rank_df,
    min_current_predicted_count=10,
    min_si_change=0.1,
    min_post_fit_observed_years=1,
    max_post_fit_si_abs_pct_error_mean=40,
    fallback_top_n=0,
)

minimal_df = interesting_df[
    [
        "pathogen",
        "antibiotic",
        "interest_score",
        "si_change",
        "current_predicted_count",
        "future_predicted_count",
        "post_fit_si_abs_pct_error_mean",
    ]
].copy()

minimal_df


In [ ]:
INTERESTING_PNG_DIR.mkdir(parents=True, exist_ok=True)

for pathogen, antibiotic_group in minimal_df[["pathogen", "antibiotic"]].values:
    result = plot_yearly_si_pair(
        si_df=si_df,
        pathogen=pathogen,
        antibiotic=antibiotic_group,
        fit_start_year=RANK_FIT_START_YEAR,
        fit_end_year=RANK_FIT_END_YEAR,
        num_additional_years=RANK_NUM_FUTURE_YEARS,
        loess_frac=0.6,
        loess_degree=2,
        tail_damping_years=None,
    )

    pathogen_slug = str(pathogen).replace("/", "_").replace(" ", "_")
    group_slug = str(antibiotic_group).replace("/", "_").replace(" ", "_")
    out_path = INTERESTING_PNG_DIR / f"{pathogen_slug}__{group_slug}.png"
    result["figure"].savefig(out_path, dpi=200, bbox_inches="tight")

print(f"Saved {len(minimal_df):,} interesting-case PNGs to {INTERESTING_PNG_DIR}")
